In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
import os
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp

spark = SparkSession.builder.getOrCreate()

# Paths
raw_path = "abfss://raw@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/"
processed_path = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/bronze/"
bronze_processed_path = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/bronze_processed/"
quarantine_path = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/quarantine/"

primary_keys = {
    "olist_customers_dataset": "customer_id",
    "olist_geolocation_dataset": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"],
    "olist_order_items_dataset": ["order_id", "order_item_id", "product_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_orders_dataset": "order_id",
    "olist_products_dataset": "product_id",
    "olist_sellers_dataset": "seller_id",
    "product_category_name_translation": "product_category_name"
}

# List CSVs
files = [f.path for f in dbutils.fs.ls(raw_path) if f.name.endswith(".csv")]

for file in files:
    # Get table name from filename
    file_name = os.path.basename(file)
    table_name = file_name.replace(".csv", "")
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    if table_name not in primary_keys:
        print(f"--> [Quarantine] No configuration found for: {table_name}")
        target_quarantine = os.path.join(quarantine_path, f"{timestamp}_{file_name}")
        dbutils.fs.mv(file, target_quarantine)
        print(f"--> Moved {file_name} to quarantine folder.")
        continue

    print(f"--- [START] Processing: {table_name}")

    # Read with robust CSV settings for Olist's messy strings
    df_raw = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "false")
              .option("multiLine", "true")
              .option("escape", '"')
              .option("quote", '"')
              .load(file))
    
    # Pk Integrity Isolation
    pk = primary_keys[table_name]
    pk_cols = pk if isinstance(pk, list) else [pk]

    null_pk_condition = " OR ".join([f"{col} IS NULL" for col in pk_cols])

    # Split into valid and invalid records
    df_invalid = df_raw.filter(null_pk_condition)
    df_valid = df_raw.filter(f"NOT ({null_pk_condition})")

    if not df_invalid.take(1):
        print(f"--> No Null PKs found in {table_name}")
    else:
        print(f"--> Null PKs found in {table_name}")
        rejected_rows_path = os.path.join(quarantine_path, "rejected", table_name)
        (df_invalid.withColumn("rejected_at", F.current_timestamp()).write.format("delta").mode("append").save(rejected_rows_path))

    # Define unique path for this specific table
    target_table_path = os.path.join(processed_path, table_name)

    # Check if table exists
    if DeltaTable.isDeltaTable(spark, target_table_path):
        print(f"--> [MERGE] Upserting data into {table_name}...")
        dt = DeltaTable.forPath(spark, target_table_path)
        pk = primary_keys[table_name]
        
        # Build merge condition for single or composite keys
        pk_cols = pk if isinstance(pk, list) else [pk]
        condition = " AND ".join([f"target.{col} = source.{col}" for col in pk_cols])

        # Deduplicate source on PK to avoid multiple matches
        df_to_upsert = df_valid.dropDuplicates(pk_cols)

        (dt.alias("target")
         .merge(df_to_upsert.alias("source"), condition)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        # Initial load: Create the table if it doesn't exist
        print(f"--> [INIT] Creating new Delta table for {table_name}...")
        (df_valid.write
         .format("delta")
         .mode("overwrite") 
         .save(target_table_path))

    # Orchestration: Move file to 'processed' only after Spark succeeds
    # target_file_path = os.path.join(bronze_processed_path, file_name)
    # dbutils.fs.mv(file, target_file_path)
    # print(f"--> [DONE] Moved {file_name} to processed folder.\n")
    print(f"--> [SUCCESS] {table_name} processed. File remains in raw for next run.\n")

message = f"Bronze Ingestion Completed"
dbutils.notebook.exit(message)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can